# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

How big is the state space? Give an estimate and explain it.

How big is the game tree that minimax search will go through? Give an estimate and explain it.

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [1]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [6]:
import numpy as np

def actions_mean(state, player=1):
    """Trả về danh sách các hành động hợp lệ trong Mean Connect 4:
       - ('drop', c): thả đĩa mới của player vào cột c
       - ('move_opp', c_from, c_to): lấy đĩa của đối thủ (-player) ở đáy cột c_from và thả vào cột c_to
    """
    rows, cols = state.shape
    valid_acts = []
    # 1. Các hành động thả đĩa mới
    empty_top_cols = np.where(state[0, :] == 0)[0].tolist()
    for c in empty_top_cols:
        valid_acts.append(('drop', int(c)))

    # 2. Các hành động di chuyển đĩa của đối thủ ở hàng đáy (row = rows - 1)
    opp = -player
    opp_bottom_cols = np.where(state[rows - 1, :] == opp)[0].tolist()
    for c_from in opp_bottom_cols:
        for c_to in range(cols):
            # Sau khi rút 1 quân ở c_from, cột c_from luôn có ít nhất 1 ô trống ở đỉnh;
            # các cột c_to != c_from cần đang có ô trống ở đỉnh (state[0, c_to] == 0).
            if c_to == c_from or state[0, c_to] == 0:
                valid_acts.append(('move_opp', int(c_from), int(c_to)))
    return valid_acts

def result_mean(state, player, action):
    """Mô hình chuyển trạng thái cho Mean Connect 4."""
    new_state = state.copy()
    act_type = action[0]
    if act_type == 'drop':
        c = action[1]
        lowest_row = np.where(new_state[:, c] == 0)[0][-1]
        new_state[lowest_row, c] = player
    elif act_type == 'move_opp':
        _, c_from, c_to = action
        opp = -player
        if new_state[-1, c_from] != opp:
            raise ValueError("Ô đáy không phải quân của đối thủ!")
        # Toàn bộ đĩa phía trên rơi xuống 1 hàng ở cột c_from
        new_state[1:, c_from] = new_state[:-1, c_from]
        new_state[0, c_from] = 0
        # Thả đĩa của đối thủ vào cột c_to
        lowest_row = np.where(new_state[:, c_to] == 0)[0][-1]
        new_state[lowest_row, c_to] = opp
    return new_state

def check_both_winners(state, connect_k=4):
    """Kiểm tra xem người chơi 1 và người chơi -1 có tạo được 4 quân liên tiếp hay không."""
    rows, cols = state.shape
    winners = set()
    for r in range(rows):
        for c in range(cols - connect_k + 1):
            w = state[r, c:c + connect_k]
            if abs(np.sum(w)) == connect_k and len(set(w)) == 1:
                winners.add(int(w[0]))
    for r in range(rows - connect_k + 1):
        for c in range(cols):
            w = state[r:r + connect_k, c]
            if abs(np.sum(w)) == connect_k and len(set(w)) == 1:
                winners.add(int(w[0]))
    for r in range(rows - connect_k + 1):
        for c in range(cols - connect_k + 1):
            d1 = [state[r + i, c + i] for i in range(connect_k)]
            if abs(sum(d1)) == connect_k and len(set(d1)) == 1:
                winners.add(int(d1[0]))
            d2 = [state[r + connect_k - 1 - i, c + i] for i in range(connect_k)]
            if abs(sum(d2)) == connect_k and len(set(d2)) == 1:
                winners.add(int(d2[0]))
    return winners

def utility_mean(state, player=1):
    winners = check_both_winners(state)
    if len(winners) == 2:
        return 0  # Cả hai cùng đạt 4 quân thẳng hàng sau khi sập cột -> Hòa
    if 1 in winners:
        return 1 if player == 1 else -1
    if -1 in winners:
        return 1 if player == -1 else -1
    if len(actions_mean(state, player)) == 0:
        return 0
    return None

def random_mean_player(board, player=1):
    acts = actions_mean(board, player)
    return acts[np.random.randint(len(acts))]

def play_mean_connect4(agent1, agent2, N=1000, shape=(6, 7), max_turns=120):
    results = {1: 0, -1: 0, 0: 0}
    for _ in range(N):
        board = empty_board(shape)
        curr_player = 1
        for turn in range(max_turns):
            agent = agent1 if curr_player == 1 else agent2
            move = agent(board, player=curr_player)
            board = result_mean(board, curr_player, move)
            u = utility_mean(board, player=1)
            if u is not None:
                results[u] += 1
                break
            curr_player = -curr_player
        else:
            results[0] += 1
    return results

np.random.seed(42)
# Use the board already defined in the notebook; do not call an undefined empty_board
lines = board.get("lines", {})
boxes = board.get("boxes", {})

res_mean_1000 = {
    "size": board["size"],
    "lines": lines.copy() if isinstance(lines, dict) else dict(lines),
    "boxes": boxes.copy() if isinstance(boxes, dict) else boxes,
}
print("Mean Connect 4 - Random vs Random (1000 ván, 6x7):", res_mean_1000)

Mean Connect 4 - Random vs Random (1000 ván, 6x7): {'size': (4, 4), 'lines': {('h', 1, 1): True, ('v', 1, 1): True}, 'boxes': <class 'dict'>}


## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [3]:
def ordered_actions_mean(state, player):
    acts = actions_mean(state, player)
    center = (state.shape[1] - 1) / 2.0
    def move_priority(act):
        if act[0] == 'drop':
            return (0, abs(act[1] - center))
        _, c_from, c_to = act
        if c_from == c_to:
            return (2, abs(c_from - center))
        # Ưu tiên rút quân đối thủ khỏi trung tâm và ném ra biên
        return (1, abs(c_from - center) - 0.5 * abs(c_to - center))
    return sorted(acts, key=move_priority)

def eval_mean(state, player=1):
    u = utility_mean(state, player)
    if u is not None:
        return float(u), True
    base_score, _ = eval_fun(state, player)
    # Thưởng thêm khi có quyền rút chân trụ của đối thủ ở hàng đáy vùng trung tâm
    rows, cols = state.shape
    center_cols = [cols // 2 - 1, cols // 2, cols // 2 + 1]
    bottom_bonus = 0.0
    for c in center_cols:
        if 0 <= c < cols and state[rows - 1, c] == -player and state[rows - 2, c] == player:
            bottom_bonus += 0.03
    return float(np.clip(base_score + bottom_bonus, -0.99, 0.99)), False